# Final trained LSTM — CPU inference gate (no retraining)

Use a standard **Colab CPU** runtime (no GPU, paid service, Drive mount or dataset). This notebook tests the actual saved final model from run_20261105T101047_013570Z. It does not call training, optimizer steps, backward, fit or the training smoke test. No inference success is asserted before execution.

Run all and select TWO local files at the single upload prompt: the preserved dist/lstm_sentiment_colab.zip and run_20261105T101047_013570Z_results.zip. Their exact SHA-256 fingerprints are pinned. A regenerated/different source ZIP is rejected intentionally. The input ZIPs/weights are private local assets and are not proposed for public publication.


## 1. Prepare the standalone checker

The embedded source below is an exact copy of tests/final_artifact_cpu_check.py. It is written only inside this remote session. Standard Colab has PyTorch; if it is absent, the checker records not_run and stops. No packages are installed on the laptop or automatically in this notebook.


In [ ]:
from pathlib import Path
import tempfile
import hashlib
import json
import subprocess
import sys
import os
from google.colab import files

os.environ['CUDA_VISIBLE_DEVICES'] = ''
WORK = Path(tempfile.mkdtemp(prefix='lstm_final_cpu_', dir='/content'))
CHECKER_SOURCE = "\"\"\"Explicit final trained-artifact CPU checks. No fit, backward, optimizer or dataset access.\n\nNot auto-discovered by unittest; run explicitly or via final_inference_cpu.ipynb.\nArchive preparation uses only the standard library. Missing PyTorch is NOT a pass.\n\"\"\"\nimport argparse\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport math\nimport os\nimport platform\nimport sys\nimport unittest\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nRUN_ID = 'run_20261105T101047_013570Z'\nSOURCE_ZIP_SHA = 'e49f872b0118bbd481c9838e17d60a9d3a894e62d39b043bbbf56e474071996d'\nRESULTS_ZIP_SHA = 'bc16b5a4c13e21495b50d7433f906dc7554bcad149037c62ca08e4bfdd2e435e'\nMODEL_SHA = '740572862066b6e39dcefd2ddafa4c77f2fbd8560590cd24a559851d65953718'\nMETADATA_SHA = 'f774d20762c5d1b8c9e18f2ceeedd7139a43a5de932984d3dd3c4487f0e27f7d'\nSOURCE_HASHES = {\n    'sentiment/__init__.py': 'e6017e636d8254480e3108f8a14ac84c39999423b5d6649c2a437f58a16fecd5',\n    'sentiment/data.py': '0ef282da837cd6f5c72cc9fe9e847cec8fbc80694cb35d4880fbc8246ed23670',\n    'sentiment/metrics.py': 'aa75019e1058a04d034a6fb982820b57f940da4841a2c8ac8e08e56280055f6c',\n    'sentiment/runtime.py': 'aac5b2fca4788fd30c10397a53252f72b6c98af588a7f8f74db274f5de68efb4',\n}\nREVIEWS = [\n    ('positive_1', 'positive', 'I loved this movie. The performances were excellent and the story was wonderful.'),\n    ('positive_2', 'positive', 'This was a brilliant film with great acting and a beautiful ending.'),\n    ('negative_1', 'negative', 'I hated this movie. The acting was terrible and the story was boring.'),\n    ('negative_2', 'negative', 'This was an awful film, a complete waste of time with bad acting.'),\n]\nCONTEXT = {}\n\n\ndef sha(raw):\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef write_report(path, report):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + '.tmp')\n    temporary.write_text(json.dumps(report, indent=2, ensure_ascii=True), encoding='utf-8')\n    temporary.replace(path)\n\n\ndef read_manifest_zip(path, expected_sha, prefix, manifest_name):\n    raw = Path(path).read_bytes()\n    if sha(raw) != expected_sha:\n        raise ValueError('Archive SHA-256 mismatch')\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)) or sum(i.file_size for i in archive.infolist()) > 100_000_000:\n            raise ValueError('Duplicate members or oversized archive')\n        for info in archive.infolist():\n            name = PurePosixPath(info.filename)\n            if (name.is_absolute() or '..' in name.parts or not name.parts\n                    or name.parts[0] != prefix or '\\\\' in info.filename\n                    or (info.external_attr >> 16) & 0o170000 == 0o120000):\n                raise ValueError('Unsafe archive path or symlink')\n        manifest = json.loads(archive.read(prefix + '/' + manifest_name))\n        expected = {prefix + '/' + name for name in manifest['files']} | {prefix + '/' + manifest_name}\n        if set(names) != expected or archive.testzip() is not None:\n            raise ValueError('Archive membership/CRC mismatch')\n        payload = {}\n        for name, item in manifest['files'].items():\n            value = archive.read(prefix + '/' + name)\n            if len(value) != item['bytes'] or sha(value) != item['sha256']:\n                raise ValueError('Member fingerprint mismatch: ' + name)\n            payload[name] = value\n        payload[manifest_name] = archive.read(prefix + '/' + manifest_name)\n    return payload\n\n\ndef prepare_inputs(source_zip, results_zip, workspace):\n    source = read_manifest_zip(source_zip, SOURCE_ZIP_SHA, 'lstm_sentiment_analysis', 'MANIFEST.json')\n    results = read_manifest_zip(results_zip, RESULTS_ZIP_SHA, RUN_ID, 'RESULTS_MANIFEST.json')\n    if json.loads(source['MANIFEST.json'])['files'] != json.loads(results['source_manifest.json'])['files']:\n        raise ValueError('Source ZIP is not the recorded experiment source')\n    for name, digest in SOURCE_HASHES.items():\n        if sha(source[name]) != digest:\n            raise ValueError('Inference source mismatch: ' + name)\n    if sha(results['model/model.pt']) != MODEL_SHA or sha(results['model/metadata.json']) != METADATA_SHA:\n        raise ValueError('Not the pinned final trained model/metadata')\n    workspace = Path(workspace)\n    workspace.mkdir(parents=True, exist_ok=False)\n    project, artifact = workspace / 'project', workspace / 'artifact'\n    for name in SOURCE_HASHES:\n        destination = project / name\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        destination.write_bytes(source[name])\n    artifact.mkdir()\n    for name in ('model.pt', 'metadata.json'):\n        (artifact / name).write_bytes(results['model/' + name])\n    # Only four inference source files and two final artifact files are extracted.\n    # No training entry point, raw data, partial weights or prior logs are extracted.\n    record = {'status': 'prepared_not_executed', 'source_zip_sha256': SOURCE_ZIP_SHA,\n              'results_zip_sha256': RESULTS_ZIP_SHA, 'project': str(project),\n              'artifact': str(artifact), 'inference_executed': False}\n    write_report(workspace / 'preparation.json', record)\n    return record\n\n\nclass FinalTrainedArtifactTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        import torch\n        from sentiment.runtime import Predictor, seed_everything\n        seed_everything(42)\n        cls.torch = torch\n        cls.predictor = Predictor.load(CONTEXT['artifact'], device='cpu')\n        cls.baseline = {key: tensor.detach().clone() for key, tensor in cls.predictor.model.state_dict().items()}\n\n    def test_01_architecture_vocabulary_preprocessing(self):\n        from dataclasses import asdict\n        from sentiment.data import PREPROCESSING\n        meta = CONTEXT['meta']\n        p = self.predictor\n        self.assertEqual(meta['architecture'], 'unidirectional-single-layer-lstm')\n        self.assertEqual(asdict(p.config), meta['config'])\n        self.assertEqual(meta['preprocessing'], PREPROCESSING)\n        self.assertEqual(meta['labels'], {'negative': 0, 'positive': 1})\n        self.assertEqual(meta['threshold'], 0.5)\n        self.assertEqual(len(p.vocab), 20000)\n        self.assertEqual(sorted(p.vocab.values()), list(range(20000)))\n        self.assertEqual((p.vocab['<PAD>'], p.vocab['<UNK>']), (0, 1))\n        expected = {'embedding.weight': (20000, 100), 'lstm.weight_ih_l0': (512, 100),\n                    'lstm.weight_hh_l0': (512, 128), 'lstm.bias_ih_l0': (512,),\n                    'lstm.bias_hh_l0': (512,), 'fc.weight': (1, 128), 'fc.bias': (1,)}\n        state = p.model.state_dict()\n        self.assertEqual({k: tuple(v.shape) for k, v in state.items()}, expected)\n        self.assertEqual(sum(v.numel() for v in state.values()), 2117889)\n        for tensor in state.values():\n            self.assertEqual(tensor.device.type, 'cpu')\n            self.assertEqual(tensor.dtype, self.torch.float32)\n            self.assertTrue(self.torch.isfinite(tensor).all().item())\n        self.assertEqual(self.torch.count_nonzero(state['embedding.weight'][0]).item(), 0)\n\n    def check_examples(self, expected_label):\n        for identifier, label, text in REVIEWS:\n            if label != expected_label:\n                continue\n            with self.subTest(example=identifier):\n                result = self.predictor.predict(text)\n                CONTEXT['report']['examples'].append({'id': identifier, 'text': text,\n                                                      'expected': label, 'actual': result})\n                probability = result['positive_probability']\n                self.assertTrue(math.isfinite(probability) and 0 <= probability <= 1)\n                self.assertEqual(result['sentiment'], 'positive' if probability >= 0.5 else 'negative')\n                self.assertEqual(result['sentiment'], label)\n\n    def test_02_positive_reviews(self):\n        self.check_examples('positive')\n\n    def test_03_negative_reviews(self):\n        self.check_examples('negative')\n\n    def test_04_empty_and_unsupported_inputs(self):\n        cases = [('empty', ''), ('whitespace', '   '), ('markup_only', '<br />!!!'),\n                 ('arabic', 'مرحبا بالعالم'), ('chinese', '你好'), ('none', None), ('number', 42)]\n        for name, value in cases:\n            with self.subTest(case=name):\n                rejected = False\n                try:\n                    self.predictor.predict(value)\n                except ValueError:\n                    rejected = True\n                CONTEXT['report']['invalid_inputs'].append({'case': name, 'rejected': rejected})\n                self.assertTrue(rejected)\n\n    def test_05_unknown_english_words(self):\n        result = self.predictor.predict('qzxvplm wurzqplx')\n        self.assertEqual(result['unknown_tokens'], 2)\n        self.assertEqual(result['tokens_used'], 2)\n        self.assertTrue(math.isfinite(result['positive_probability']))\n        CONTEXT['report']['unknown_word_result'] = result\n\n    def test_06_predictions_stable_after_reload(self):\n        from sentiment.runtime import Predictor\n        restored = Predictor.load(CONTEXT['artifact'], device='cpu')\n        for identifier, _, text in REVIEWS:\n            with self.subTest(example=identifier):\n                first = self.predictor.predict(text)\n                repeated = self.predictor.predict(text)\n                reloaded = restored.predict(text)\n                delta = abs(first['positive_probability'] - reloaded['positive_probability'])\n                CONTEXT['report']['reload_checks'].append({'id': identifier, 'before': first,\n                    'repeated': repeated, 'after_reload': reloaded, 'absolute_delta': delta})\n                self.assertEqual(first, repeated)\n                self.assertEqual(first['sentiment'], reloaded['sentiment'])\n                self.assertLessEqual(delta, 1e-7)\n\n    def test_07_padding_and_single_batch(self):\n        from sentiment.data import encode\n        t = self.torch\n        p = self.predictor\n        ids, length = encode(REVIEWS[0][2], p.vocab, p.config.max_length)\n        with t.inference_mode():\n            original = p.model(t.tensor([ids]), t.tensor([length]))\n            padded = p.model(t.tensor([ids + [0] * 8]), t.tensor([length]))\n        CONTEXT['report']['forward_executed'] = True\n        self.assertEqual(tuple(original.shape), (1,))\n        t.testing.assert_close(original, padded, rtol=0, atol=1e-7)\n\n    def test_99_no_weights_or_files_changed(self):\n        self.assertFalse(self.predictor.model.training)\n        for key, value in self.predictor.model.state_dict().items():\n            self.assertTrue(self.torch.equal(value, self.baseline[key]), key)\n        self.assertTrue(all(p.grad is None for p in self.predictor.model.parameters()))\n        self.assertEqual(sha((Path(CONTEXT['artifact']) / 'model.pt').read_bytes()), MODEL_SHA)\n        self.assertEqual(sha((Path(CONTEXT['artifact']) / 'metadata.json').read_bytes()), METADATA_SHA)\n\n\ndef run_checks(project, artifact, report_path):\n    report = {'status': 'not_run', 'inference_executed': False, 'retraining': False,\n              'created_utc': datetime.now(timezone.utc).isoformat(), 'run_id': RUN_ID,\n              'model_sha256': MODEL_SHA, 'metadata_sha256': METADATA_SHA,\n              'source_sha256': SOURCE_HASHES, 'checker_sha256': sha(Path(__file__).read_bytes()),\n              'examples': [], 'invalid_inputs': [], 'reload_checks': [], 'tests_run': 0}\n    try:\n        project, artifact = Path(project).resolve(), Path(artifact).resolve()\n        for name, expected in SOURCE_HASHES.items():\n            if sha((project / name).read_bytes()) != expected:\n                raise ValueError('Pinned inference source mismatch: ' + name)\n        if sha((artifact / 'model.pt').read_bytes()) != MODEL_SHA or sha((artifact / 'metadata.json').read_bytes()) != METADATA_SHA:\n            raise ValueError('Final artifact fingerprint mismatch')\n        report['artifact_integrity'] = 'passed'\n        meta = json.loads((artifact / 'metadata.json').read_text())\n        report.update(model_config=meta['config'], preprocessing=meta['preprocessing'],\n                      vocabulary_size=len(meta['vocabulary']))\n        if importlib.util.find_spec('torch') is None:\n            report['reason'] = 'PyTorch unavailable; final inference NOT executed. Use Colab CPU.'\n            write_report(report_path, report)\n            print(report['reason'])\n            return 2\n        os.environ['CUDA_VISIBLE_DEVICES'] = ''\n        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')\n        sys.path.insert(0, str(project))\n        import torch\n        import numpy as np\n        torch.set_num_threads(1)\n        report['environment'] = {'python': platform.python_version(), 'torch': torch.__version__,\n                                 'numpy': np.__version__, 'device': 'cpu', 'threads': 1}\n        CONTEXT.update(artifact=artifact, meta=meta, report=report)\n        suite = unittest.TestLoader().loadTestsFromTestCase(FinalTrainedArtifactTests)\n        result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)\n        report.update(inference_executed=bool(report['examples'] or report.get('unknown_word_result')\n                                            or report.get('forward_executed')), tests_run=result.testsRun,\n                      failures=len(result.failures), errors=len(result.errors), skipped=len(result.skipped),\n                      failure_details=[{'test': str(test), 'traceback': trace} for test, trace in result.failures + result.errors])\n        passed = result.wasSuccessful() and not result.skipped and result.testsRun == 8\n        report['status'] = 'passed' if passed else 'failed'\n        write_report(report_path, report)\n        return 0 if passed else 1\n    except Exception as error:\n        report.update(status='failed', error_type=type(error).__name__, error=str(error))\n        write_report(report_path, report)\n        print(type(error).__name__ + ': ' + str(error))\n        return 1\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--prepare', action='store_true')\n    parser.add_argument('--source-zip', type=Path)\n    parser.add_argument('--results-zip', type=Path)\n    parser.add_argument('--workspace', type=Path)\n    parser.add_argument('--project', type=Path, default=Path(__file__).resolve().parents[1])\n    parser.add_argument('--artifact', type=Path)\n    parser.add_argument('--report', type=Path, default=Path('outputs/final_inference/report.json'))\n    args = parser.parse_args()\n    if args.prepare:\n        if not all((args.source_zip, args.results_zip, args.workspace)):\n            parser.error('--prepare needs --source-zip, --results-zip and a new --workspace')\n        print(json.dumps(prepare_inputs(args.source_zip, args.results_zip, args.workspace), indent=2))\n        return 0\n    if args.artifact is None:\n        parser.error('--artifact is required for final inference checks')\n    return run_checks(args.project, args.artifact, args.report)\n\n\nif __name__ == '__main__':\n    raise SystemExit(main())\n"
CHECKER = WORK / 'final_artifact_cpu_check.py'
CHECKER.write_text(CHECKER_SOURCE, encoding='utf-8')
print('Checker SHA-256:', hashlib.sha256(CHECKER.read_bytes()).hexdigest())


## 2. Upload both ZIPs; verify and prepare actual model

Only four inference source files and the selected model.pt plus exact metadata.json are extracted. No dataset, training entry point or partial checkpoint is extracted. Every ZIP member hash is checked first, and the source manifest must equal the real experiment source manifest. A mismatch stops execution before weight loading.


In [ ]:
os.chdir(WORK)
uploaded = files.upload()
expected = {
    'e49f872b0118bbd481c9838e17d60a9d3a894e62d39b043bbbf56e474071996d': 'source',
    'bc16b5a4c13e21495b50d7433f906dc7554bcad149037c62ca08e4bfdd2e435e': 'results',
}
selected = {}
if len(uploaded) != 2:
    raise ValueError('Upload exactly the preserved source ZIP and original results ZIP')
for name, content in uploaded.items():
    digest = hashlib.sha256(content).hexdigest()
    if digest not in expected or expected[digest] in selected:
        raise ValueError('Unexpected or duplicate ZIP fingerprint')
    selected[expected[digest]] = WORK / name
del uploaded
INPUTS = WORK / 'verified_inputs'
prepared = subprocess.run([sys.executable, '-B', str(CHECKER), '--prepare',
    '--source-zip', str(selected['source']), '--results-zip', str(selected['results']),
    '--workspace', str(INPUTS)], capture_output=True, text=True)
print(prepared.stdout + prepared.stderr)
prepared.check_returncode()


## 3. Execute CPU-only inference and download real evidence

Eight tests cover metadata/config/vocabulary/tensor shapes, two authored positive and two negative reviews, empty/unsupported inputs, unknown English words, repeated predictions and fresh reload (absolute tolerance 1e-7), padding invariance and unchanged weights/files. Expected labels are diagnostic cases, not guaranteed outcomes: a misclassification is recorded as failure, never hidden or fixed by retraining/tuning. They are not a new test-set accuracy estimate.

The report and complete log are downloaded even on a failed/not_run check when available. Only status=passed, 8 tests, zero failures/errors/skips and inference_executed=true closes this gate. If PyTorch is unavailable, select a standard CPU runtime with it preinstalled and rerun; no GPU is needed. Return the downloaded evidence ZIP for review.


In [ ]:
REPORT = WORK / 'final_inference_report.json'
LOG = WORK / 'final_inference.log'
completed = subprocess.run([sys.executable, '-B', str(CHECKER),
    '--project', str(INPUTS / 'project'), '--artifact', str(INPUTS / 'artifact'),
    '--report', str(REPORT)], capture_output=True, text=True)
LOG.write_text(completed.stdout + completed.stderr, encoding='utf-8')
print(LOG.read_text())
if not REPORT.exists():
    raise RuntimeError('No report produced; this is not a passed gate')
result = json.loads(REPORT.read_text())
print(json.dumps(result, indent=2))
import zipfile
EVIDENCE = WORK / 'final_inference_cpu_evidence.zip'
payload = {p.name: p.read_bytes() for p in [REPORT, LOG, INPUTS / 'preparation.json']}
manifest = {'status': result['status'], 'checker_sha256': hashlib.sha256(CHECKER.read_bytes()).hexdigest(),
            'files': {name: {'bytes': len(raw), 'sha256': hashlib.sha256(raw).hexdigest()} for name, raw in payload.items()}}
with zipfile.ZipFile(EVIDENCE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name, raw in payload.items():
        archive.writestr(name, raw)
    archive.writestr('EVIDENCE_MANIFEST.json', json.dumps(manifest, indent=2))
with zipfile.ZipFile(EVIDENCE) as archive:
    if archive.testzip() is not None:
        raise RuntimeError('Evidence ZIP CRC failed')
files.download(str(EVIDENCE))
completed.check_returncode()
assert result['status'] == 'passed' and result['tests_run'] == 8
assert result['inference_executed'] is True and result['retraining'] is False
assert all(result[k] == 0 for k in ('failures', 'errors', 'skipped'))
print('Final trained-artifact CPU inference gate PASSED in this execution.')


## Evidence and scope

Output: final_inference_cpu_evidence.zip contains report, log, input-preparation record and hashes; it contains no weights, vocabulary list, raw dataset or input ZIPs. A self-produced hash manifest verifies integrity, not a trusted signature. This notebook does not validate data redistribution rights or grant a code license. The original 86.29% and improved 86.09% / 0.8607 experiment scores stay unchanged.
